## Extract the Data

In [1]:
import pandas as pd

url = "https://transparencia.sns.gov.pt/explore/dataset/formacao-medica-medicos-recem-especialistas/download?format=csv&timezone=Europe/Berlin&use_labels_for_header=true"

df = (
    pd.read_csv(url, sep=";").astype({
        "Nº Registo": "int64",
        "Período": "int64",
        "Médicos Recém-Especialistas": "int64"
    })
)
df.head()


,Nº Registo,Período,Especialidade,Instituição de Formação,Médicos Recém-Especialistas
0,2,2020,Anatomia Patológica,Centro Hospitalar Universitário de Lisboa Nort...,1
1,11,2020,Anestesiologia,"Centro Hospitalar do Baixo Vouga, E.P.E. - Hos...",1
2,22,2020,Anestesiologia,Centro Hospitalar Universitário de Lisboa Cent...,1
3,25,2020,Anestesiologia,"Centro Hospitalar e Universitário de Coimbra, ...",8
4,26,2020,Anestesiologia,Hospital Divino Espírito Santo de Ponta Delgad...,2


## Transform the Data

In [2]:
import numpy as np

# normalizar strings para 'nan'
df = df.replace(
    ["", " ", "NA", "N/A", "NULL", "-"],
    np.nan
)

In [3]:
def validar_dados(df):
    erros = []

    anos_invalidos = df[
        (df["Período"] < 2000) |
        (df["Período"] > 2030)
    ]

    if not anos_invalidos.empty:
        erros.append(
            f"Existem {len(anos_invalidos)} anos inválidos"
        )

    negativos = df[
        df["Médicos Recém-Especialistas"] < 0
    ]

    if not negativos.empty:
        erros.append(
            f"Existem {len(negativos)} valores negativos"
        )

    # check for missing values
    missing_values = df.isna().sum()
    if missing_values.any():
        erros.append(
            f"Existem {missing_values.sum()} valores em falta"
        )

    return erros

print(validar_dados(df))

[]


In [4]:
# Tratamento de Dados
df.info()
df.describe(include="all")

<class 'pandas.DataFrame'>
RangeIndex: 4629 entries, 0 to 4628
Data columns (total 5 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   Nº Registo                   4629 non-null   int64
 1   Período                      4629 non-null   int64
 2   Especialidade                4629 non-null   str  
 3   Instituição de Formação      4629 non-null   str  
 4   Médicos Recém-Especialistas  4629 non-null   int64
dtypes: int64(3), str(2)
memory usage: 465.9 KB


,Nº Registo,Período,Especialidade,Instituição de Formação,Médicos Recém-Especialistas
count,4629.000000,4629.000000,4629,4629,4629.000000
unique,NaN,NaN,50,260,NaN
top,NaN,NaN,Medicina Geral e Familiar,"Centro Hospitalar e Universitário de Coimbra, ...",NaN
freq,NaN,NaN,754,161,NaN
mean,2315.000000,2022.683733,NaN,NaN,2.019011
std,1336.421528,1.746152,NaN,NaN,2.177135
min,1.000000,2020.000000,NaN,NaN,1.000000
25%,1158.000000,2021.000000,NaN,NaN,1.000000
50%,2315.000000,2023.000000,NaN,NaN,1.000000
75%,3472.000000,2024.000000,NaN,NaN,2.000000


In [5]:
for col in df.columns:
    print(f"\n=== {col} ===")
    print(df[col].value_counts(dropna=False).head(20))


=== Nº Registo ===
Nº Registo
2      1
11     1
22     1
25     1
26     1
33     1
36     1
39     1
41     1
44     1
60     1
67     1
74     1
78     1
83     1
85     1
86     1
89     1
97     1
108    1
Name: count, dtype: int64

=== Período ===
Período
2025    994
2024    802
2021    776
2022    707
2023    685
2020    665
Name: count, dtype: int64

=== Especialidade ===
Especialidade
Medicina Geral e Familiar            754
Medicina Interna                     323
Pediatria                            220
Cirurgia Geral                       210
Ortopedia                            198
Psiquiatria                          168
Anestesiologia                       163
Ginecologia / Obstetrícia            159
Saúde Pública                        159
Cardiologia                          135
Patologia Clínica                    128
Medicina Física e de Reabilitação    115
Pneumologia                          114
Gastrenterologia                     111
Oncologia Médica             

In [6]:
df["Especialidade"].unique()

<ArrowStringArray>
[                      'Anatomia Patológica',
                            'Anestesiologia',
                               'Cardiologia',
                         'Cirurgia Cardíaca',
                            'Cirurgia Geral',
                       'Cirurgia Pediátrica',
                         'Cirurgia Vascular',
                       'Dermatovenereologia',
                   'Endocrinologia/Nutrição',
                             'Estomatologia',
                          'Gastrenterologia',
                           'Genética Médica',
                 'Ginecologia / Obstetrícia',
                       'Hematologia Clínica',
                          'Imunoalergologia',
                       'Doenças Infecciosas',
                      'Medicina do Trabalho',
         'Medicina Física e de Reabilitação',
                 'Medicina Geral e Familiar',
                          'Medicina Interna',
                          'Medicina Nuclear',
               

In [7]:
df["Instituição de Formação"].value_counts()

Instituição de Formação
Centro Hospitalar e Universitário de Coimbra, E.P.E.         161
Centro Hospitalar Universitário de São João, E.P.E.          155
Centro Hospitalar Universitário de Lisboa Central, E.P.E.    122
Centro Hospitalar de Lisboa Ocidental, E.P.E.                117
Unidade Local de Saúde de São João, E.P.E.                   104
                                                            ... 
CUF Porto                                                      1
CS Funchal Zona I - Bom Jesus                                  1
CS Funchal Zona I - Santa Isabel/Monte                         1
CS Santa Cruz - Caniço                                         1
Hospital CUF Tejo                                              1
Name: count, Length: 260, dtype: int64

## Connect to DB

In [8]:
import sqlite3

In [9]:
df_medicos = df.copy()

df_medicos = df_medicos.rename(columns={
    "Nº Registo": "id",
    "Período": "ano",
    "Instituição de Formação": "instituicao_formacao",
    "Médicos Recém-Especialistas": "quantidade"
})

# tabela ESPECIALIDADES
specialties = df_medicos[['Especialidade']].drop_duplicates().reset_index(drop=True)
specialties['id'] = specialties.index + 1

# tabela INSTITUIÇÃO
institutions = df_medicos[['instituicao_formacao']].drop_duplicates().reset_index(drop=True)
institutions['id'] = institutions.index + 1

print(specialties.head())
print(institutions.head())


         Especialidade  id
0  Anatomia Patológica   1
1       Anestesiologia   2
2          Cardiologia   3
3    Cirurgia Cardíaca   4
4       Cirurgia Geral   5
                                instituicao_formacao  id
0  Centro Hospitalar Universitário de Lisboa Nort...   1
1  Centro Hospitalar do Baixo Vouga, E.P.E. - Hos...   2
2  Centro Hospitalar Universitário de Lisboa Cent...   3
3  Centro Hospitalar e Universitário de Coimbra, ...   4
4  Hospital Divino Espírito Santo de Ponta Delgad...   5


In [11]:
conn = sqlite3.connect("medicos.db")
cursor = conn.cursor()

# ---------------------------
# 1. INSERIR ESPECIALIDADES
# ---------------------------
for linha in specialties.values:
    especialidade = linha[0]
    id_especialidade = linha[1]

    cursor.execute("""
        INSERT OR IGNORE INTO ESPECIALIDADES (id, nome)
        VALUES (?, ?)
    """, (id_especialidade, especialidade))

# ---------------------------
# 2. INSERIR INSTITUIÇÕES
# ---------------------------
for linha in institutions.values:
    instituicao = linha[0]
    id_instituicao = linha[1]

    cursor.execute("""
        INSERT OR IGNORE INTO INSTITUICAO (id, nome)
        VALUES (?, ?)
    """, (id_instituicao, instituicao))


conn.commit()
conn.close()

print("Dados inseridos com sucesso nas tabelas ESPECIALIDADES e INSTITUIÇÃO.")

Dados inseridos com sucesso nas tabelas ESPECIALIDADES e INSTITUIÇÃO.


In [17]:
df_medicos

,id,ano,Especialidade,instituicao_formacao,quantidade
0,2,2020,Anatomia Patológica,Centro Hospitalar Universitário de Lisboa Nort...,1
1,11,2020,Anestesiologia,"Centro Hospitalar do Baixo Vouga, E.P.E. - Hos...",1
2,22,2020,Anestesiologia,Centro Hospitalar Universitário de Lisboa Cent...,1
3,25,2020,Anestesiologia,"Centro Hospitalar e Universitário de Coimbra, ...",8
4,26,2020,Anestesiologia,Hospital Divino Espírito Santo de Ponta Delgad...,2
...,...,...,...,...,...
4624,4607,2025,Saúde Pública,"Unidade Local de Saúde de Lisboa Ocidental, E....",2
4625,4613,2025,Saúde Pública,"Unidade Local de Saúde da Região de Aveiro, E....",1
4626,4615,2025,Saúde Pública,"Unidade Local de Saúde da Região de Leiria, E....",3
4627,4621,2025,Urologia,"Unidade Local de Saúde do Algarve, E.P.E.",1


In [ ]:
# Conectar à DB
conn = sqlite3.connect("medicos.db")
cursor = conn.cursor()

# ---------------------------
# 3. OBTER IDs
# ---------------------------
spec_df = pd.read_sql_query("SELECT * FROM ESPECIALIDADES", conn)
inst_df = pd.read_sql_query("SELECT * FROM INSTITUICAO", conn)

print(spec_df)

# Juntar IDs ao dataset
df_medicos = df_medicos.merge(spec_df, left_on="Especialidade", right_on="nome")
df_medicos = df_medicos.merge(inst_df, left_on="instituicao_formacao", right_on="nome")

conn.close()

df_medicos

    id                                       nome
0    1                        Anatomia Patológica
1    2                             Anestesiologia
2    3                                Cardiologia
3    4                          Cirurgia Cardíaca
4    5                             Cirurgia Geral
5    6                        Cirurgia Pediátrica
6    7                          Cirurgia Vascular
7    8                        Dermatovenereologia
8    9                    Endocrinologia/Nutrição
9   10                              Estomatologia
10  11                           Gastrenterologia
11  12                            Genética Médica
12  13                  Ginecologia / Obstetrícia
13  14                        Hematologia Clínica
14  15                           Imunoalergologia
15  16                        Doenças Infecciosas
16  17                       Medicina do Trabalho
17  18          Medicina Física e de Reabilitação
18  19                  Medicina Geral e Familiar


,id_x,ano,Especialidade,instituicao_formacao,quantidade,id_y,nome_x,id,nome_y
0,2,2020,Anatomia Patológica,Centro Hospitalar Universitário de Lisboa Nort...,1,1,Anatomia Patológica,1,Centro Hospitalar Universitário de Lisboa Nort...
1,11,2020,Anestesiologia,"Centro Hospitalar do Baixo Vouga, E.P.E. - Hos...",1,2,Anestesiologia,2,"Centro Hospitalar do Baixo Vouga, E.P.E. - Hos..."
2,22,2020,Anestesiologia,Centro Hospitalar Universitário de Lisboa Cent...,1,2,Anestesiologia,3,Centro Hospitalar Universitário de Lisboa Cent...
3,25,2020,Anestesiologia,"Centro Hospitalar e Universitário de Coimbra, ...",8,2,Anestesiologia,4,"Centro Hospitalar e Universitário de Coimbra, ..."
4,26,2020,Anestesiologia,Hospital Divino Espírito Santo de Ponta Delgad...,2,2,Anestesiologia,5,Hospital Divino Espírito Santo de Ponta Delgad...
...,...,...,...,...,...,...,...,...,...
4624,4607,2025,Saúde Pública,"Unidade Local de Saúde de Lisboa Ocidental, E....",2,36,Saúde Pública,138,"Unidade Local de Saúde de Lisboa Ocidental, E...."
4625,4613,2025,Saúde Pública,"Unidade Local de Saúde da Região de Aveiro, E....",1,36,Saúde Pública,157,"Unidade Local de Saúde da Região de Aveiro, E...."
4626,4615,2025,Saúde Pública,"Unidade Local de Saúde da Região de Leiria, E....",3,36,Saúde Pública,137,"Unidade Local de Saúde da Região de Leiria, E...."
4627,4621,2025,Urologia,"Unidade Local de Saúde do Algarve, E.P.E.",1,37,Urologia,134,"Unidade Local de Saúde do Algarve, E.P.E."


In [19]:
# Conectar à DB
conn = sqlite3.connect("medicos.db")
cursor = conn.cursor()

# ---------------------------
# 2. INSERIR FORMACOES
# ---------------------------
for linha in df_medicos.values:
    id = linha[0]
    ano = linha[1]
    id_especialidade = linha[5]
    id_instituicao = linha[7]
    quantidade = linha[4]

    cursor.execute("""
        INSERT OR IGNORE INTO FORMACAO (id, ano, id_especialidade, id_instituicao, quantidade)
        VALUES (?, ?, ?, ?, ?)
    """, (id, ano, id_especialidade, id_instituicao, quantidade))


conn.commit()
conn.close()

## PostgreSQL

In [17]:
import psycopg2

conn = psycopg2.connect(
    dbname="medical formation",
    user="postgres",
    password="postgres159",
    host="localhost",
    port="5432"
)

cursor = conn.cursor()

# ---------------------------
# 1. INSERIR ESPECIALIDADES
# ---------------------------
for linha in specialties.values:
    especialidade = linha[0]
    id_especialidade = linha[1]

    cursor.execute("""
        INSERT INTO "ESPECIALIDADES" (id, nome)
        VALUES (%s, %s)
        ON CONFLICT (id) DO NOTHING;
    """, (id_especialidade, especialidade))

# ---------------------------
# 2. INSERIR INSTITUIÇÕES
# ---------------------------
for linha in institutions.values:
    instituicao = linha[0]
    id_instituicao = linha[1]

    cursor.execute("""
        INSERT INTO "INSTITUICAO" (id, nome)
        VALUES (%s, %s)
        ON CONFLICT (id) DO NOTHING;
    """, (id_instituicao, instituicao))

conn.commit()
cursor.close()
conn.close()


In [18]:
import psycopg2
import pandas as pd

# Conectar à DB (PostgreSQL)
conn = psycopg2.connect(
    dbname="medical formation",
    user="postgres",
    password="postgres159",
    host="localhost",
    port="5432"
)

# ---------------------------
# 3. OBTER IDs
# ---------------------------
spec_df = pd.read_sql_query("SELECT * FROM \"ESPECIALIDADES\"", conn)
inst_df = pd.read_sql_query("SELECT * FROM \"INSTITUICAO\"", conn)

print(spec_df)

# Juntar IDs ao dataset
df_medicos = df_medicos.merge(spec_df, left_on="Especialidade", right_on="nome")
df_medicos = df_medicos.merge(inst_df, left_on="instituicao_formacao", right_on="nome")

conn.close()

df_medicos

    id                                       nome
0    1                        Anatomia Patológica
1    2                             Anestesiologia
2    3                                Cardiologia
3    4                          Cirurgia Cardíaca
4    5                             Cirurgia Geral
5    6                        Cirurgia Pediátrica
6    7                          Cirurgia Vascular
7    8                        Dermatovenereologia
8    9                    Endocrinologia/Nutrição
9   10                              Estomatologia
10  11                           Gastrenterologia
11  12                            Genética Médica
12  13                  Ginecologia / Obstetrícia
13  14                        Hematologia Clínica
14  15                           Imunoalergologia
15  16                        Doenças Infecciosas
16  17                       Medicina do Trabalho
17  18          Medicina Física e de Reabilitação
18  19                  Medicina Geral e Familiar


C:\Users\joaot\AppData\Local\Temp\ipykernel_20236\4196562879.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  spec_df = pd.read_sql_query("SELECT * FROM \"ESPECIALIDADES\"", conn)
C:\Users\joaot\AppData\Local\Temp\ipykernel_20236\4196562879.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  inst_df = pd.read_sql_query("SELECT * FROM \"INSTITUICAO\"", conn)


,id_x,ano,Especialidade,instituicao_formacao,quantidade,id_y,nome_x,id,nome_y
0,2,2020,Anatomia Patológica,Centro Hospitalar Universitário de Lisboa Nort...,1,1,Anatomia Patológica,1,Centro Hospitalar Universitário de Lisboa Nort...
1,11,2020,Anestesiologia,"Centro Hospitalar do Baixo Vouga, E.P.E. - Hos...",1,2,Anestesiologia,2,"Centro Hospitalar do Baixo Vouga, E.P.E. - Hos..."
2,22,2020,Anestesiologia,Centro Hospitalar Universitário de Lisboa Cent...,1,2,Anestesiologia,3,Centro Hospitalar Universitário de Lisboa Cent...
3,25,2020,Anestesiologia,"Centro Hospitalar e Universitário de Coimbra, ...",8,2,Anestesiologia,4,"Centro Hospitalar e Universitário de Coimbra, ..."
4,26,2020,Anestesiologia,Hospital Divino Espírito Santo de Ponta Delgad...,2,2,Anestesiologia,5,Hospital Divino Espírito Santo de Ponta Delgad...
...,...,...,...,...,...,...,...,...,...
4624,4607,2025,Saúde Pública,"Unidade Local de Saúde de Lisboa Ocidental, E....",2,36,Saúde Pública,138,"Unidade Local de Saúde de Lisboa Ocidental, E...."
4625,4613,2025,Saúde Pública,"Unidade Local de Saúde da Região de Aveiro, E....",1,36,Saúde Pública,157,"Unidade Local de Saúde da Região de Aveiro, E...."
4626,4615,2025,Saúde Pública,"Unidade Local de Saúde da Região de Leiria, E....",3,36,Saúde Pública,137,"Unidade Local de Saúde da Região de Leiria, E...."
4627,4621,2025,Urologia,"Unidade Local de Saúde do Algarve, E.P.E.",1,37,Urologia,134,"Unidade Local de Saúde do Algarve, E.P.E."


In [19]:
import psycopg2

# Conectar à DB
conn = psycopg2.connect(
    dbname="medical formation",
    user="postgres",
    password="postgres159",
    host="localhost",
    port="5432"
)

cursor = conn.cursor()

# ---------------------------
# 2. INSERIR FORMACOES
# ---------------------------
for linha in df_medicos.values:
    id = linha[0]
    ano = linha[1]
    id_especialidade = linha[5]
    id_instituicao = linha[7]
    quantidade = linha[4]

    cursor.execute("""
        INSERT INTO "FORMACAO" (id, ano, id_especialidade, id_instituicao, quantidade)
        VALUES (%s, %s, %s, %s, %s)
        ON CONFLICT (id) DO NOTHING;
    """, (id, ano, id_especialidade, id_instituicao, quantidade))

conn.commit()
cursor.close()
conn.close()
